In [0]:
from pyspark.sql.functions import *

bronze_base = "abfss://bronze@travelappprojectstorage.dfs.core.windows.net"

In [0]:
trip_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip"
)

trip_schedule_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip_schedule"
)

tourist_df = spark.read.format("parquet").load(
    f"{bronze_base}/tourist"
)

trip_request_places_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip_request_places"
)

tourist_place_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip_places"
)

latest_location_df = spark.read.format("parquet").load(
    f"{bronze_base}/latest_trilateration"
)

In [0]:
trip_df.printSchema()
trip_schedule_df.printSchema()
tourist_df.printSchema()
trip_request_places_df.printSchema()
tourist_place_df.printSchema()
latest_location_df.printSchema()

In [0]:
print(trip_df.count())
print(trip_schedule_df.count())
print(tourist_df.count())
print(trip_request_places_df.count())
print(tourist_place_df.count())
print(latest_location_df.count())

In [0]:
latest_location_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in latest_location_df.columns
]).display()

In [0]:
trip_df.groupBy("TripId").count().filter(
    col("count") > 1
).display()
trip_schedule_df.groupBy("ScheduleId").count().filter(
    col("count") > 1
).display()
latest_location_df.groupBy("TouristId").count().filter(
    col("count") > 1
).display()

In [0]:
stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        "abfss://bronze@travelappprojectstorage.dfs.core.windows.net/_schemas/trip_adherence"
    )
    .load(
        f"{bronze_base}/latest_trilateration"
    )
)

In [0]:
display(
    stream_df,
    checkpointLocation="abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/trip_adherence_bronze"
)